# **Load Libraries**

In [ ]:
import pandas as pd
import numpy as np
from oddt.fingerprints import PLEC
import oddt
from joblib import Parallel, delayed
from tqdm import tqdm
import glob
import os
import tempfile
from openbabel import openbabel
from rdkit import Chem
from rdkit.Chem import AllChem
import deepchem as dc
from deepchem.utils.vina_utils import prepare_inputs
from deepchem.utils import download_url, load_from_disk
from deepchem.feat import RdkitGridFeaturizer

import sys, os
sys.path.insert(0, os.path.join("/home/juni/working/mettl3/notebooks/attention_score/AttentionScore_materials_importable/AttentionScore/", "src"))


# **Generate plec features**

In [ ]:
from attentionscore.features.plec import plec_from_dir

df_plec = plec_from_dir(
    docked_dir="path/to/docked_sdf",       # folder with *.sdf
    protein_path="path/to/receptor.pdb",   # PDB/MOL2 etc.
    n_jobs=20,
    size=4092,
    depth_protein=4,
    depth_ligand=2,
    distance_cutoff=4.5,
    sparse=False,
    sort_numeric=True,                     # mimics your numeric filename sort
    output="numpy",                        # or "base64"/"bytes"
)
df_plec.head()


# **Generate ECFP4 features**

In [ ]:
from attentionscore.features.fingerprints import ecfp4_dataframe

# df has columns: ID, SMILES_STD
df_ecfp = ecfp4_dataframe(df, smiles_col="SMILES_STD", id_col="ID",
                          n_bits=2048, radius=2, drop_invalid=True,
                          output="base64")  # or "numpy" / "bitvect"


# **Generate Avalon fingerprints features**

In [ ]:
from attentionscore.features.fingerprints import calculate_avalon_array

smiles = df["SMILES_STD"].astype(str).tolist()
X_avalon = np.vstack([calculate_avalon_array(smi, nBits=512) for smi in smiles])  # (N, 512)